In [1]:
import os
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchinfo import summary
from torch import nn    
from torch.nn import functional as F
from torch.utils.data import random_split
import numpy as np

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Model Architecture

In [3]:
class UNET(nn.Module):
    def __init__(self, in_channels=1, out_channels=2, features=(16, 32, 64, 128)):
        super().__init__()

        self.encoders = nn.ModuleList()
        current = in_channels
        for feature in features:
            self.encoders.append(self.conv_block(current, feature))
            current = feature

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = nn.Sequential(
            self.conv_block(features[-1], features[-1] * 2),
            nn.Dropout2d(0.2)
            )

        self.up_convs = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        
        for feature in reversed(features):
            self.up_convs.append(self.up_conv(feature * 2, feature))
            self.dec_blocks.append(self.conv_block(feature * 2, feature))

        self.final = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=out_channels),
            nn.LeakyReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=out_channels),
            nn.LeakyReLU(inplace=True),
        )

    def up_conv(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.LeakyReLU(inplace=True),
        )

    def forward(self, x):
        original = x
        skips = []

        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for up, dec, skip in zip(self.up_convs, self.dec_blocks, reversed(skips)):
            x = up(x)           
            x = torch.cat((skip, x), dim=1)
            x = dec(x)

        mask = torch.sigmoid(self.final(x))
        return mask * original

In [4]:
model = UNET(in_channels=1, out_channels=2).to(device)
summary(model, input_size=(6, 1, 1040, 416), device=device)

Layer (type:depth-idx)                   Output Shape              Param #
UNET                                     [6, 2, 1040, 416]         --
├─ModuleList: 1-7                        --                        (recursive)
│    └─Sequential: 2-1                   [6, 16, 1040, 416]        --
│    │    └─Conv2d: 3-1                  [6, 16, 1040, 416]        144
│    │    └─GroupNorm: 3-2               [6, 16, 1040, 416]        32
│    │    └─LeakyReLU: 3-3               [6, 16, 1040, 416]        --
│    │    └─Conv2d: 3-4                  [6, 16, 1040, 416]        2,304
│    │    └─GroupNorm: 3-5               [6, 16, 1040, 416]        32
│    │    └─LeakyReLU: 3-6               [6, 16, 1040, 416]        --
├─MaxPool2d: 1-2                         [6, 16, 520, 208]         --
├─ModuleList: 1-7                        --                        (recursive)
│    └─Sequential: 2-2                   [6, 32, 520, 208]         --
│    │    └─Conv2d: 3-7                  [6, 32, 520, 208]     

Data Loading

In [5]:
DATA_DIR = "../datasets/spectrograms"

In [6]:
class SpectrogramDataset(Dataset):
    def __init__(self, directory):
        self.directory = directory
        self.mixture_files = sorted([f for f in os.listdir(directory) if f.endswith('_mixture.npy')])

    def __len__(self):
        return len(self.mixture_files)

    def __getitem__(self, idx):

        mixture_path = os.path.join(self.directory, self.mixture_files[idx])

        vocal_path = mixture_path.replace('_mixture.npy', '_vocals.npy')

        instrumental_path = mixture_path.replace('_mixture.npy', '_instrumental.npy')

        mixture = np.load(mixture_path)
        vocals = np.load(vocal_path)
        instrumental = np.load(instrumental_path)

        X = torch.from_numpy(mixture).float().unsqueeze(0)

        Y = torch.from_numpy(
            np.stack([vocals, instrumental], axis=0)
        ).float()

        X = F.pad(X, (0, 15, 0, 15))
        Y = F.pad(Y, (0, 15, 0, 15))

        return X, Y

In [7]:
train_dataset = SpectrogramDataset(os.path.join(DATA_DIR,"train"))
val_dataset = SpectrogramDataset(os.path.join(DATA_DIR,"val"))

print(len(train_dataset))
print(len(val_dataset))

6896
981


In [8]:
train_loader = DataLoader(
    train_dataset,
    batch_size=6,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=6,
    shuffle=False
)

Traning Loop

In [9]:
class SpectrogramLoss(nn.Module):
    def __init__(self, eps=1e-7):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):

        pred = pred.float()
        target = target.float()

        diff = pred - target

        numerator = torch.linalg.vector_norm(diff, dim=(2, 3))

        denominator = torch.linalg.vector_norm(target, dim=(2, 3))

        denominator = denominator.clamp_min(1.0)

        sc_loss = (numerator / denominator).mean()

        log_loss = torch.abs(
            torch.log(pred + self.eps) -
            torch.log(target + self.eps)
        ).mean()

        return sc_loss + log_loss

In [10]:
epochs = 50
patience = 6
lastbest = 0
best_loss_value = float("inf")

loss_fn = SpectrogramLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.75, patience=3, min_lr=1e-5)

train_loss_epochwise = []
val_loss_epochwise = []

MODEL_DIR = "../models/"
model_name = "vocal_inst_sep_unet"

PLOT_DIR = "../plots/"
os.makedirs(MODEL_DIR, exist_ok=True)

In [11]:
def save_loss_plots():
    
    plt.plot(train_loss_epochwise)
    plt.title("epochwise Train Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.savefig(os.path.join(PLOT_DIR, f"{model_name}_train_loss_epochwise.png"))
    plt.close()

    plt.plot(val_loss_epochwise)
    plt.title("epochwise Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.savefig(os.path.join(PLOT_DIR, f"{model_name}_val_loss_epochwise.png"))
    plt.close()

In [ ]:
for epoch in range(epochs):

    model.train()
    running_train_loss = 0.0

    for batch_idx, (X, Y) in enumerate(train_loader):
        X = X.to(device)
        Y = Y.to(device)

        optimizer.zero_grad()

        prediction = model(X)

        loss = loss_fn(prediction,Y)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] Step [{batch_idx+1}/{len(train_loader)}] Train Loss: {loss.item():.4f}")

    avg_train_loss = running_train_loss / len(train_loader)
    train_loss_epochwise.append(avg_train_loss)

    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for X, Y in val_loader:
            X = X.to(device)
            Y = Y.to(device)

            prediction = model(X)
        
            loss = loss_fn(prediction,Y)

            running_val_loss += loss.item()

    avg_val_loss = running_val_loss / len(val_loader)
    val_loss_epochwise.append(avg_val_loss)

    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    save_loss_plots()
    
    if avg_val_loss < best_loss_value:
        best_loss_value = avg_val_loss
        lastbest = epoch

        torch.save(model.state_dict(), os.path.join(MODEL_DIR, f"{model_name}_best.pt"))
        print(f"New best model saved (val loss: {best_loss_value:.4f})")
        
    elif epoch - lastbest >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

    scheduler.step(avg_val_loss)

    
torch.save(model.state_dict(), os.path.join(MODEL_DIR, f"{model_name}_last.pt"))

Epoch [1/50] Step [1/1150] Train Loss: 17.6031
Epoch [1/50] Step [51/1150] Train Loss: 2.4109
Epoch [1/50] Step [101/1150] Train Loss: 2.8905
Epoch [1/50] Step [151/1150] Train Loss: 6.0608
Epoch [1/50] Step [201/1150] Train Loss: 11.9491
Epoch [1/50] Step [251/1150] Train Loss: 2.8027
Epoch [1/50] Step [301/1150] Train Loss: 4.3525
Epoch [1/50] Step [351/1150] Train Loss: 2.1784
Epoch [1/50] Step [401/1150] Train Loss: 3.0081
Epoch [1/50] Step [451/1150] Train Loss: 3.5256
Epoch [1/50] Step [501/1150] Train Loss: 2.9557
Epoch [1/50] Step [551/1150] Train Loss: 2.5813
Epoch [1/50] Step [601/1150] Train Loss: 7.3990
Epoch [1/50] Step [651/1150] Train Loss: 4.2476
Epoch [1/50] Step [701/1150] Train Loss: 4.5801
Epoch [1/50] Step [751/1150] Train Loss: 4.1336
Epoch [1/50] Step [801/1150] Train Loss: 4.5337
Epoch [1/50] Step [851/1150] Train Loss: 3.4654
Epoch [1/50] Step [901/1150] Train Loss: 3.1678
Epoch [1/50] Step [951/1150] Train Loss: 2.8024
Epoch [1/50] Step [1001/1150] Train Loss:

In [ ]:
model = UNET(in_channels=1, out_channels=2)

In [ ]:
state_dict = torch.load("../models/vocal_inst_sep_unet_best.pt", map_location=device)
model.load_state_dict(state_dict)
model.eval()

dummy_input = torch.randn(1, 1, 1040, 416)

torch.onnx.export(model, dummy_input, "../models/vocal_inst_sep.onnx", dynamo=False)

C:\Users\vallu\AppData\Local\Temp\ipykernel_10220\2736576058.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy_input, "../models/vocal_inst_sep.onnx", dynamo=False)
